In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/datathon-2026-round-1/products.csv
/kaggle/input/competitions/datathon-2026-round-1/sample_submission.csv
/kaggle/input/competitions/datathon-2026-round-1/promotions.csv
/kaggle/input/competitions/datathon-2026-round-1/shipments.csv
/kaggle/input/competitions/datathon-2026-round-1/order_items.csv
/kaggle/input/competitions/datathon-2026-round-1/reviews.csv
/kaggle/input/competitions/datathon-2026-round-1/inventory.csv
/kaggle/input/competitions/datathon-2026-round-1/returns.csv
/kaggle/input/competitions/datathon-2026-round-1/sales.csv
/kaggle/input/competitions/datathon-2026-round-1/orders.csv
/kaggle/input/competitions/datathon-2026-round-1/geography.csv
/kaggle/input/competitions/datathon-2026-round-1/customers.csv
/kaggle/input/competitions/datathon-2026-round-1/baseline.ipynb
/kaggle/input/competitions/datathon-2026-round-1/payments.csv
/kaggle/input/competitions/datathon-2026-round-1/web_traffic.csv


# 1. Thiết lập đường dẫn

In [2]:
import pandas as pd
import numpy as np

DATA_DIR = '/kaggle/input/competitions/datathon-2026-round-1/' 

print("Đã thiết lập xong đường dẫn!")

Đã thiết lập xong đường dẫn!


# 2. Định dạng lại các columns

In [3]:
def load_and_fix_date(file_name):
    df = pd.read_csv(DATA_DIR + file_name)
    for col in df.columns:
        if any(x in col.lower() for x in ['date', 'day', 'ngày']):
            df = df.rename(columns={col: 'date'})
            break
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return df

sales = load_and_fix_date('sales.csv')
traffic = load_and_fix_date('web_traffic.csv')
promo = load_and_fix_date('promotions.csv')

print("Bước 2 xong! Đã load và sửa tên cột tự động.")

Bước 2 xong! Đã load và sửa tên cột tự động.


# 3. Merge bảng

In [4]:
df_ml = sales.merge(traffic, on='date', how='left')
df_ml = df_ml.merge(promo, on='date', how='left')

df_ml = df_ml.fillna(0)

print("Kích thước bảng cuối cùng:", df_ml.shape)
display(df_ml.head(10)) 

Kích thước bảng cuối cùng: (3833, 18)


,date,Revenue,COGS,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source,promo_id,promo_name,promo_type,discount_value,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,2012-07-04,5123547.94,3982991.19,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
1,2012-07-05,2751773.45,2150580.23,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
2,2012-07-06,3054029.42,2517632.84,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
3,2012-07-07,2667930.94,2108246.62,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
4,2012-07-08,2360851.90,1808622.79,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
5,2012-07-09,3548386.46,2787841.68,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
6,2012-07-10,5234938.62,4044438.84,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
7,2012-07-11,5582884.78,4338313.07,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
8,2012-07-12,5734632.02,4458811.27,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0
9,2012-07-13,5309511.71,4143402.78,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0,0,0,0.0,0.0


# Câu 1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [5]:
orders = pd.read_csv(DATA_DIR + 'orders.csv')

for col in orders.columns:
    if any(x in col.lower() for x in ['date', 'day', 'ngày']):
        orders = orders.rename(columns={col: 'date'})
        break

for col in orders.columns:
    if any(x in col.lower() for x in ['customer', 'khách', 'cust_id']):
        orders = orders.rename(columns={col: 'customer_id'})
        break

orders['date'] = pd.to_datetime(orders['date'], errors='coerce')

orders = orders.sort_values(['customer_id', 'date'])

orders['inter_order_gap'] = orders.groupby('customer_id')['date'].diff().dt.days

all_gaps = orders['inter_order_gap'].dropna()

median_result = all_gaps.median()

print(f"--- KẾT QUẢ ---")
print(f"Trung vị số ngày giữa hai lần mua liên tiếp là: {median_result} ngày")

--- KẾT QUẢ ---
Trung vị số ngày giữa hai lần mua liên tiếp là: 144.0 ngày


# Câu 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [6]:
products = pd.read_csv(DATA_DIR + 'products.csv')

products.columns = products.columns.str.lower().str.strip()

products['margin'] = (products['price'] - products['cogs']) / products['price']

segment_margin = products.groupby('segment')['margin'].mean().reset_index()

segment_margin = segment_margin.sort_values(by='margin', ascending=False)

print("--- KẾT QUẢ TỶ SUẤT LỢI NHUẬN THEO PHÂN KHÚC ---")
display(segment_margin)

best_segment = segment_margin.iloc[0]['segment']
best_value = segment_margin.iloc[0]['margin'] * 100

print(f"\n=> Phân khúc có tỷ suất lợi nhuận cao nhất là: {best_segment} ({best_value:.2f}%)")

--- KẾT QUẢ TỶ SUẤT LỢI NHUẬN THEO PHÂN KHÚC ---


,segment,margin
6,Standard,0.313442
5,Premium,0.285377
1,All-weather,0.284176
0,Activewear,0.265600
4,Performance,0.263650
2,Balanced,0.258038
7,Trendy,0.240758
3,Everyday,0.236343



=> Phân khúc có tỷ suất lợi nhuận cao nhất là: Standard (31.34%)


# Câu 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [7]:
returns = pd.read_csv(DATA_DIR + 'returns.csv')
products = pd.read_csv(DATA_DIR + 'products.csv')

returns.columns = returns.columns.str.lower().str.strip()
products.columns = products.columns.str.lower().str.strip()

df_returns_products = returns.merge(products, on='product_id', how='left')

streetwear_returns = df_returns_products[df_returns_products['category'] == 'Streetwear']

reason_counts = streetwear_returns['return_reason'].value_counts()

print("--- THỐNG KÊ LÝ DO TRẢ HÀNG (STREETWEAR) ---")
print(reason_counts)

if not reason_counts.empty:
    top_reason = reason_counts.idxmax()
    print(f"\n=> Lý do trả hàng xuất hiện nhiều nhất là: {top_reason}")
else:
    print("\n=> Không tìm thấy dữ liệu trả hàng cho danh mục Streetwear.")

--- THỐNG KÊ LÝ DO TRẢ HÀNG (STREETWEAR) ---
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

=> Lý do trả hàng xuất hiện nhiều nhất là: wrong_size


# Câu 4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [8]:
web_traffic = pd.read_csv(DATA_DIR + "web_traffic.csv")

web_traffic.columns = web_traffic.columns.str.lower().str.strip()

source_bounce = traffic.groupby('traffic_source')['bounce_rate'].mean().reset_index()

source_bounce = source_bounce.sort_values(by='bounce_rate', ascending=True)

print("--- TỶ LỆ THOÁT TRUNG BÌNH THEO NGUỒN TRUY CẬP ---")
display(source_bounce)

best_source = source_bounce.iloc[0]['traffic_source']
lowest_rate = source_bounce.iloc[0]['bounce_rate']

print(f"\n=> Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: {best_source} ({lowest_rate:.2f})")

--- TỶ LỆ THOÁT TRUNG BÌNH THEO NGUỒN TRUY CẬP ---


,traffic_source,bounce_rate
1,email_campaign,0.004458
5,social_media,0.004476
3,paid_search,0.004478
4,referral,0.004499
2,organic_search,0.004504
0,direct,0.004511



=> Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: email_campaign (0.00)


# Câu 5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [9]:
items = pd.read_csv(DATA_DIR + 'order_items.csv')

items.columns = items.columns.str.lower().str.strip()

promo_col = None
for col in items.columns:
    if 'promo' in col:
        promo_col = col
        break

if promo_col:
    total_rows = len(items)
    promo_applied_rows = items[promo_col].notna().sum()
    percentage = (promo_applied_rows / total_rows) * 100
    
    print(f"--- KẾT QUẢ ---")
    print(f"Tổng số dòng: {total_rows}")
    print(f"Số dòng có khuyến mãi: {promo_applied_rows}")
    print(f"Tỷ lệ phần trăm xấp xỉ: {percentage:.2f}%")
else:
    print("Không tìm thấy cột khuyến mãi!")

--- KẾT QUẢ ---
Tổng số dòng: 714669
Số dòng có khuyến mãi: 276316
Tỷ lệ phần trăm xấp xỉ: 38.66%


/tmp/ipykernel_16/3877888350.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  items = pd.read_csv(DATA_DIR + 'order_items.csv')


# Câu 6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [10]:
customers = pd.read_csv(DATA_DIR + 'customers.csv')
orders = pd.read_csv(DATA_DIR + 'orders.csv')

customers.columns = customers.columns.str.lower().str.strip()
orders.columns = orders.columns.str.lower().str.strip()

df_combined = orders.merge(customers[['customer_id', 'age_group']], on='customer_id', how='left')

df_combined = df_combined.dropna(subset=['age_group'])

total_orders_per_group = df_combined.groupby('age_group')['order_id'].count()

valid_customers = customers.dropna(subset=['age_group'])
num_customers_per_group = valid_customers.groupby('age_group')['customer_id'].nunique()

avg_orders_per_customer = (total_orders_per_group / num_customers_per_group).reset_index()
avg_orders_per_customer.columns = ['age_group', 'avg_orders']

avg_orders_per_customer = avg_orders_per_customer.sort_values(by='avg_orders', ascending=False)

print("--- THỐNG KÊ SỐ ĐƠN TRUNG BÌNH THEO NHÓM TUỔI ---")
display(avg_orders_per_customer)

best_group = avg_orders_per_customer.iloc[0]['age_group']
max_val = avg_orders_per_customer.iloc[0]['avg_orders']

print(f"\n=> Nhóm tuổi có số đơn hàng trung bình cao nhất là: {best_group} ({max_val:.2f} đơn/khách)")

--- THỐNG KÊ SỐ ĐƠN TRUNG BÌNH THEO NHÓM TUỔI ---


,age_group,avg_orders
4,55+,5.406851
3,45-54,5.357241
2,35-44,5.337343
1,25-34,5.245226
0,18-24,5.226656



=> Nhóm tuổi có số đơn hàng trung bình cao nhất là: 55+ (5.41 đơn/khách)


# Câu 7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [11]:
geography = pd.read_csv(DATA_DIR + "geography.csv")
payments = pd.read_csv(DATA_DIR + "payments.csv")

orders_geo = orders.merge(geography[['zip', 'region']], on='zip', how='left')
pay_region  = payments.merge(orders_geo[['order_id', 'region']], on='order_id')
rev_region  = pay_region.groupby('region')['payment_value'].sum().sort_values(ascending=False)
print('Tổng doanh thu theo vùng (payment_value):')
print(rev_region.apply(lambda x: f'{x:,.0f}').to_string())

print('Vùng có doanh thu cao nhất là: ', rev_region.idxmax())

Tổng doanh thu theo vùng (payment_value):
region
East       7,291,150,819
Central    4,719,491,268
West       3,670,227,178
Vùng có doanh thu cao nhất là:  East


# Câu 8: Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [12]:
orders = pd.read_csv(DATA_DIR + 'orders.csv')

orders.columns = orders.columns.str.lower().str.strip()

cancelled_orders = orders[orders['order_status'].str.lower() == 'cancelled']
payment_counts = cancelled_orders['payment_method'].value_counts()

print("--- THỐNG KÊ PHƯƠNG THỨC THANH TOÁN CỦA ĐƠN HÀNG BỊ HỦY ---")
print(payment_counts)

if not payment_counts.empty:
    top_payment = payment_counts.idxmax()
    print(f"\n=> Phương thức thanh toán được sử dụng nhiều nhất trong các đơn bị hủy là: {top_payment}")
else:
    print("\n=> Không tìm thấy đơn hàng nào có trạng thái 'cancelled'.")

--- THỐNG KÊ PHƯƠNG THỨC THANH TOÁN CỦA ĐƠN HÀNG BỊ HỦY ---
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

=> Phương thức thanh toán được sử dụng nhiều nhất trong các đơn bị hủy là: credit_card


# Câu 9: Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [13]:
returns = pd.read_csv(DATA_DIR + 'returns.csv')
products = pd.read_csv(DATA_DIR + 'products.csv')
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')

for df in [order_items, returns, products]:
    df.columns = df.columns.str.lower().str.strip()

df_returns_products = returns.merge(products, on='product_id', how='left')
cnt_size_return = df_returns_products['size'].value_counts()

df_items_products = order_items.merge(products, on='product_id', how='left')
cnt_size_sold = df_items_products['size'].value_counts()

percentage = (cnt_size_return / cnt_size_sold)

percentage = percentage.loc[['S', 'M', 'L', 'XL']]
print(percentage)

if not percentage.empty:
    top_percentage = percentage.idxmax()
    print(top_percentage)    

/tmp/ipykernel_16/3475113077.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(DATA_DIR + 'order_items.csv')


size
S     0.056515
M     0.055660
L     0.056250
XL    0.055200
Name: count, dtype: float64
S


# Câu 10: Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [14]:
payments = pd.read_csv(DATA_DIR + 'payments.csv')

payments.columns = payments.columns.str.lower().str.strip()

order_totals = payments.groupby(['order_id', 'installments'])['payment_value'].sum().reset_index()

plan_avg = order_totals.groupby('installments')['payment_value'].mean().reset_index()

plan_avg = plan_avg.sort_values(by='payment_value', ascending=False)

best_plan = int(plan_avg.iloc[0]['installments'])
highest_avg = plan_avg.iloc[0]['payment_value']

print(plan_avg)
print(f"\n=> Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: {best_plan} tháng (Trung bình: {highest_avg:.2f})")


   installments  payment_value
3             6   24446.654403
2             3   24399.635486
4            12   24245.772694
0             1   24113.274166
1             2     708.473729

=> Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: 6 tháng (Trung bình: 24446.65)
